In [ ]:
import torch

from dataset import BillingualDataset, compute_casual_mask
from vanilla_transformer import build_transformer
from train import get_model, get_ds

from config import get_config, get_weigths_file_path

from datasets import load_dataset
from tokenizers import Tokenizer

from tqdm import tqdm
from pathlib import Path

c:\Users\hellg\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)

Max length of source sentence 45
Max length of source sentence 41


In [18]:
def translate(sentence: str):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    config = get_config()
    tokenizer_src = Tokenizer.from_file(
        str(Path(config['tokenizer_file'].format(config['lang_src'])))
        )
    tokenizer_tgt = Tokenizer.from_file(
        str(Path(config['tokenizer_file'].format(config['lang_tgt'])))
        )
    model = build_transformer(tokenizer_src.get_vocab_size(),
                              tokenizer_tgt.get_vocab_size(),
                              config['seq_len'],
                              config['seq_len'],
                              config['embedding_dim']).to(device)

    model_filename = get_weigths_file_path(config, '19')
    state_dict = torch.load(model_filename)
    model.load_state_dict(state_dict['model_state_dict'])
    seq_len = config['seq_len']

    model.eval()
    with torch.no_grad():
        source = tokenizer_src.encode(sentence)
        source = torch.cat([
            torch.tensor([tokenizer_src.token_to_id('<SOS>')], dtype=torch.int64),
            torch.tensor(source.ids, dtype=torch.int64),
            torch.tensor([tokenizer_src.token_to_id('<EOS>')], dtype=torch.int64),
            torch.tensor([tokenizer_src.token_to_id('<PAD>')] * (seq_len - len(source.ids) - 2), dtype=torch.int64)
        ], dim=0).to(device)
        source_mask = (source != tokenizer_src.token_to_id('<PAD>')).unsqueeze(0).unsqueeze(0).to(device)
        encoder_output = model.encode(source, source_mask)

        decoder_input = torch.empty(1, 1).fill_(tokenizer_tgt.token_to_id('<SOS>')).type_as(source).to(device)

        while decoder_input.shape[1] < seq_len:

            decoder_mask = compute_casual_mask(decoder_input.shape[1]).type_as(source_mask).to(device)
            decoder_output = model.decode(encoder_output, source_mask, decoder_input, decoder_mask) # [1, 1, E]

            probs = model.projection(decoder_output[:, -1])

            _, next_word = torch.max(probs, dim=1)
            decoder_input = torch.cat(
                [
                    decoder_input,
                    torch.empty(1,1).type_as(source).fill_(next_word.item()).to(device)
                ], dim=1
            )

            if next_word == tokenizer_tgt.token_to_id('<EOS>'):
                break
    return tokenizer_tgt.decode(decoder_input[0].tolist())


In [45]:
translate('Eine Frau, die in einer Küche eine Schale mit Essen hält.')

'A woman in a kitchen holding a bowl of food .'